In [ ]:
# ============================================================
# COLAB USERS ONLY — Uncomment and run this cell
# Local VS Code users: SKIP this cell
# ============================================================

# !pip install langchain langchain-groq langchain-google-genai langchain-community python-dotenv duckduckgo-search -q

# from google.colab import userdata
# import os
# os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
# os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
# print("Colab setup complete!")

# Day 4: LangChain Framework Deep Dive

**Agentic AI Hands-On Course** | Dr. Kanthi Kiran Sirra | Sr. AI Engineer

**What you'll build today:**
1. Prompt Templates — reusable, validated prompts with variables
2. Output Parsers — get structured data (JSON, Pydantic) from the LLM
3. LCEL pipe chains — the modern `prompt | llm | parser` pattern
4. Sequential chains — multi-step pipelines
5. RunnableBranch — route inputs to different chains
6. LangChain agents — `create_tool_calling_agent` + `AgentExecutor`

**Key transition:** From Smolagents (Days 1-2) to LangChain (Day 4+).
Same concepts, more powerful framework.

---

## 0. Setup — Load your API keys

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

groq_key = os.getenv("GROQ_API_KEY", "")
print(f"Groq API Key: {'Loaded' if len(groq_key) > 10 else 'MISSING'}")

In [ ]:
# Create our LLM — reused throughout the session
from langchain_groq import ChatGroq

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0, max_tokens=500)

# Quick test
response = llm.invoke("Say 'LangChain session started!' in exactly 4 words.")
print(response.content)

---
## 1. Prompt Templates — Why not just use f-strings?

In Days 1-3, you wrote prompts as plain strings. That works for demos, but in production you need:
- **Validation** — what if a variable is None?
- **Reusability** — same template, different inputs
- **Composability** — plug prompts into chains

LangChain's `ChatPromptTemplate` solves all three.

In [ ]:
# === PROVE IT: 3 problems with f-strings ===

# Problem 1: No validation — None becomes the string "None"
company = None
customer = "Ravi"
question = "Where is my order?"

prompt_fstring = f"You are a support agent for {company}. Help {customer} with: {question}"
print("Problem 1 — No validation:")
print(f"  Prompt sent to LLM: '{prompt_fstring}'")
print(f"  BUG: 'for None' — the LLM gets nonsense and nobody gets an error!")
print()

# Problem 2: Can't reuse — the f-string evaluates IMMEDIATELY
company = "Flipkart"
prompt1 = f"You are a support agent for {company}."
company = "Amazon"  # change the variable
print("Problem 2 — Can't reuse:")
print(f"  prompt1 is still: '{prompt1}'")
print(f"  Changing 'company' after the f-string does nothing — it was already evaluated.")
print(f"  You'd have to copy-paste the entire f-string again for Amazon.")
print()

# Problem 3: Can't swap/version — prompt is buried in code
print("Problem 3 — Can't swap or version:")
print("  The prompt is hardcoded inside your Python code.")
print("  To change the prompt, you change the code. To A/B test, you duplicate code.")
print("  No way to version prompts separately from your application logic.")

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# Create the template ONCE — it's a reusable object, not an immediately-evaluated string
support_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a support agent for {company}. Be helpful and empathetic."),
    ("human", "Customer {customer} asks: {question}")
])

# SOLUTION 1: Validation — missing variables raise a clear error
print("Solution 1 — Validation:")
try:
    # Try to use the template with a missing variable
    support_prompt.invoke({"company": "Flipkart", "customer": "Ravi"})
    # 'question' is missing!
except Exception as e:
    print(f"  Missing variable caught: {type(e).__name__}")
    print(f"  You get a clear error BEFORE anything is sent to the LLM.")
    print(f"  f-string would silently send 'None' — template STOPS you.")
print()

# SOLUTION 2: Reusable — same template, fill with different values anytime
print("Solution 2 — Reusable:")
for company in ["Flipkart", "Amazon", "Swiggy"]:
    formatted = support_prompt.invoke({
        "company": company,
        "customer": "Ravi",
        "question": "Where is my order?"
    })
    # Just show the system message to prove it changes
    print(f"  {company}: {formatted.messages[0].content}")
print(f"  Same template object, 3 different companies — no copy-paste!")
print()

# SOLUTION 3: Swappable — store templates as objects, swap without changing logic
print("Solution 3 — Swappable / Versionable:")
prompt_v1 = ChatPromptTemplate.from_messages([
    ("system", "You are a support agent for {company}. Be formal and professional."),
    ("human", "{question}")
])
prompt_v2 = ChatPromptTemplate.from_messages([
    ("system", "You are a friendly helper at {company}. Use casual language and emojis."),
    ("human", "{question}")
])

# Swap prompts without changing ANY other code
active_prompt = prompt_v2  # just change this one line to switch versions!
result = active_prompt.invoke({"company": "Flipkart", "question": "Where is my order?"})
print(f"  Active prompt (v2): {result.messages[0].content}")
print(f"  To switch to v1: just change 'active_prompt = prompt_v1' — no other code changes!")
print(f"  You can A/B test, version control, and swap prompts independently from your app.")

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# Create a reusable prompt template with variables
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a {role}. Be {style}."),
    ("human", "{question}")
])

# See what the template looks like
print("Template variables:", prompt.input_variables)
print()

# Format it with specific values
formatted = prompt.invoke({
    "role": "senior AI engineer",
    "style": "technical and precise",
    "question": "What is RAG in 2 sentences?"
})

print("Formatted messages:")
for msg in formatted.messages:
    print(f"  {msg.type}: {msg.content}")

In [ ]:
# The power of templates — SAME template, DIFFERENT inputs

personas = [
    {"role": "5-year-old child", "style": "super simple with fun examples"},
    {"role": "senior AI engineer", "style": "technical and precise"},
    {"role": "business executive", "style": "focused on ROI and business impact"},
]

for persona in personas:
    result = llm.invoke(prompt.invoke({
        **persona,
        "question": "What is an AI agent?"
    }))
    print(f"As {persona['role']}:")
    print(f"  {result.content[:150]}...")
    print()

**Key insight:** Same template, 3 different personas, 3 completely different answers. The template is the reusable structure — the variables are what change.

### 1.1 Quick reference — Prompt Template types

| Method | When to use |
|---|---|
| `ChatPromptTemplate.from_messages([...])` | Multi-message (system + human) — **use this 95% of the time** |
| `ChatPromptTemplate.from_template("...")` | Single human message only |
| `PromptTemplate.from_template("...")` | Old-style (no roles) — avoid this |

---

## 2. Output Parsers — Get structured data from the LLM

Without parsers, the LLM returns free text. Sometimes a list, sometimes a paragraph, sometimes with markdown. Unpredictable.

Output parsers solve this by:
1. Telling the LLM what format to use (via format instructions injected into the prompt)
2. Parsing the response into a Python data structure

### 2.1 StrOutputParser — The simplest one

In [ ]:
from langchain_core.output_parsers import StrOutputParser

# Without parser — you get an AIMessage object
raw = llm.invoke("What is LangChain in one sentence?")
print(f"Without parser: {type(raw).__name__}")
print(f"  Content: {raw.content}")
print()

# With StrOutputParser — you get a clean string
parser = StrOutputParser()
clean = parser.invoke(raw)
print(f"With parser: {type(clean).__name__}")
print(f"  Content: {clean}")

### 2.2 JsonOutputParser — Get a Python dictionary

In [ ]:
from langchain_core.output_parsers import JsonOutputParser

json_parser = JsonOutputParser()

json_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Always respond in valid JSON format only. No markdown, no explanation, no backticks."),
    ("human", "List 3 popular AI frameworks with their company and main use case. {format_instructions}")
])

# Build the chain: prompt -> llm -> parser
chain = json_prompt | llm | json_parser

result = chain.invoke({"format_instructions": json_parser.get_format_instructions()})

print(f"Type: {type(result).__name__}")
print()

# Now you can use it as a Python dict/list!
if isinstance(result, list):
    for item in result:
        print(f"  - {item}")
elif isinstance(result, dict):
    for key, val in result.items():
        print(f"  {key}: {val}")

### 2.3 PydanticOutputParser — Type-safe structured output

In [ ]:
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from typing import List

# Define the EXACT structure you want
class AgentAnalysis(BaseModel):
    agent_name: str = Field(description="Name of the AI agent or framework")
    strengths: List[str] = Field(description="List of 2-3 key strengths")
    best_use_case: str = Field(description="The single best use case")
    complexity_score: int = Field(description="Complexity from 1-10, where 1 is simplest")

pydantic_parser = PydanticOutputParser(pydantic_object=AgentAnalysis)

pydantic_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an AI expert. Respond ONLY with valid JSON matching the schema. No markdown, no backticks."),
    ("human", "Analyze the LangChain framework.\n\n{format_instructions}")
])

chain = pydantic_prompt | llm | pydantic_parser

result = chain.invoke({"format_instructions": pydantic_parser.get_format_instructions()})

print(f"Type: {type(result).__name__}")
print(f"Name: {result.agent_name}")
print(f"Strengths: {result.strengths}")
print(f"Best use case: {result.best_use_case}")
print(f"Complexity: {result.complexity_score}/10")

**Why Pydantic matters:** In production, you don't want to hope the LLM gives you the right fields. Pydantic *validates* the output — if a field is missing or the wrong type, you get a clear error instead of a silent bug.

---

## 3. LCEL — The Pipe Operator

This is the most important concept in modern LangChain. Instead of wrapping everything in chain classes (the old way), you connect components with the `|` pipe operator.

```
chain = prompt | llm | parser
```

Read left to right: prompt formats input -> LLM generates response -> parser extracts clean output.

Like Unix: `cat file.txt | grep "error" | sort | head -5`

In [ ]:
# The simplest LCEL chain — 3 components piped together
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a {role}. Be concise — max 2 sentences."),
    ("human", "{question}")
])

# Build the chain with the pipe operator
chain = prompt | llm | StrOutputParser()

# Use it — just pass the variables as a dict
result = chain.invoke({
    "role": "AI tutor",
    "question": "What is the difference between an agent and a chatbot?"
})

print(f"Type: {type(result).__name__}")  # str, not AIMessage!
print(f"Answer: {result}")

In [ ]:
# Same chain, different inputs — the chain is reusable
questions = [
    {"role": "AI tutor", "question": "What is LCEL?"},
    {"role": "AI tutor", "question": "Why did LangChain deprecate LLMChain?"},
    {"role": "AI tutor", "question": "What is the ReAct pattern?"},
]

for q in questions:
    answer = chain.invoke(q)
    print(f"Q: {q['question']}")
    print(f"A: {answer}")
    print()

---
## 4. Sequential Chains — Multi-step pipelines

Real applications need multiple LLM calls in sequence. Example:
1. **Step 1:** Summarize a long text
2. **Step 2:** Translate the summary to another language

The output of Step 1 becomes the input to Step 2.

### 4.1 Two-step chain: Summarize then Translate

In [ ]:
from langchain_core.runnables import RunnableLambda

# Step 1: Summarize
summarize_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert summarizer. Summarize the following text in exactly 2 sentences."),
    ("human", "{text}")
])
summarize_chain = summarize_prompt | llm | StrOutputParser()

# Step 2: Translate
translate_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a translator. Translate the following text to {language}. Output ONLY the translation."),
    ("human", "{text}")
])
translate_chain = translate_prompt | llm | StrOutputParser()

# Test Step 1 alone
long_text = """Artificial intelligence agents represent a paradigm shift in how we build software. 
Unlike traditional chatbots that simply respond to queries, agents can autonomously decide which 
tools to use, plan multi-step solutions, and learn from their observations. The key components 
that make agents powerful include: tool calling (the ability to use external functions), memory 
(maintaining context across interactions), reasoning (the ReAct pattern of think-act-observe), 
and planning (breaking complex tasks into manageable steps). Frameworks like LangChain, CrewAI, 
and LangGraph provide the infrastructure to build these agents efficiently."""

summary = summarize_chain.invoke({"text": long_text})
print("=== Step 1: Summary ===")
print(summary)
print()

# Now feed the summary into Step 2
translation = translate_chain.invoke({"text": summary, "language": "Hindi"})
print("=== Step 2: Hindi Translation ===")
print(translation)

### 4.2 Connecting them into ONE chain

Instead of calling each step manually, we can wire them together using `RunnableLambda` as the glue:

In [ ]:
# Full pipeline: text -> summarize -> translate

full_chain = (
    summarize_chain
    | RunnableLambda(lambda summary: {"text": summary, "language": "Hindi"})
    | translate_chain
)

# One call does both steps!
result = full_chain.invoke({"text": long_text})
print("=== Full pipeline (Summarize + Translate in one call) ===")
print(result)

In [ ]:
# Change the language by swapping the glue function

telugu_chain = (
    summarize_chain
    | RunnableLambda(lambda summary: {"text": summary, "language": "Telugu"})
    | translate_chain
)

result = telugu_chain.invoke({"text": long_text})
print("=== Summarize + Translate to Telugu ===")
print(result)

**What just happened:**
1. `summarize_chain` takes `{"text": ...}` and returns a string (the summary)
2. `RunnableLambda` converts that string into `{"text": summary, "language": "Telugu"}` — the format `translate_chain` expects
3. `translate_chain` takes that dict and returns the translated string

**The RunnableLambda is the glue** — it transforms the output shape of one step into the input shape the next step expects.

---

## 5. RunnableBranch — Route inputs to different chains

Sometimes you don't want a linear pipeline. You want to route different questions to different chains.

Example: A support bot that sends billing questions to one chain and technical questions to another.

In [ ]:
from langchain_core.runnables import RunnableBranch

# Create specialized chains for different topics
billing_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a billing support specialist. Help with payment, pricing, and invoice questions. Be empathetic and helpful. Max 3 sentences."),
    ("human", "{question}")
])
billing_chain = billing_prompt | llm | StrOutputParser()

technical_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a technical support engineer. Help with bugs, errors, and technical issues. Be precise and give actionable steps. Max 3 sentences."),
    ("human", "{question}")
])
technical_chain = technical_prompt | llm | StrOutputParser()

general_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a friendly general support agent. Help with any question. Max 3 sentences."),
    ("human", "{question}")
])
general_chain = general_prompt | llm | StrOutputParser()

print("3 specialized chains created!")

In [ ]:
# Create the router — checks conditions in order, falls back to default
billing_keywords = ["bill", "price", "payment", "invoice", "charge", "refund"]
technical_keywords = ["error", "bug", "crash", "broken", "not working", "fix"]

router = RunnableBranch(
    # (condition, chain) pairs — checked in order
    (lambda x: any(w in x["question"].lower() for w in billing_keywords),
     billing_chain),
    
    (lambda x: any(w in x["question"].lower() for w in technical_keywords),
     technical_chain),
    
    # Default fallback — if no condition matches
    general_chain
)

print("Router ready! Let's test...")

In [ ]:
# Test the router with different question types
test_questions = [
    "Why was I charged twice on my invoice?",                  # -> billing
    "My app keeps crashing with a memory error",               # -> technical
    "What are your business hours?",                           # -> general
    "Can I get a refund for last month?",                      # -> billing
    "The API returns a 500 error when I send a POST request",  # -> technical
]

for q in test_questions:
    # Detect which chain will be used
    if any(w in q.lower() for w in billing_keywords):
        route = "BILLING"
    elif any(w in q.lower() for w in technical_keywords):
        route = "TECHNICAL"
    else:
        route = "GENERAL"
    
    answer = router.invoke({"question": q})
    print(f"[{route}] Q: {q}")
    print(f"A: {answer}")
    print()

**Notice:** Same `router.invoke()` call, but different questions get routed to different specialized chains automatically. The billing question gets empathetic billing language. The technical question gets precise debugging steps.

---

## 6. LangChain Tools — The LangChain way to define tools

You already know `@tool` from Smolagents (Day 2). LangChain has its own `@tool` — same concept, different import.

In [ ]:
from langchain_core.tools import tool

@tool
def calculate(expression: str) -> str:
    """Evaluates a mathematical expression and returns the result.
    Use this for any math calculations.
    
    Args:
        expression: A mathematical expression like '2 + 2' or '(15 * 3) / 4.5'
    """
    try:
        result = eval(expression)
        return str(result)
    except Exception as e:
        return f"Error: {e}"

@tool
def search_web(query: str) -> str:
    """Searches the web for current information using DuckDuckGo.
    Use this when you need up-to-date information.
    
    Args:
        query: The search query
    """
    try:
        from duckduckgo_search import DDGS
        results = DDGS().text(query, max_results=3)
        return "\n\n".join([f"{r['title']}: {r['body']}" for r in results])
    except Exception as e:
        return f"Search error: {e}"

@tool
def get_word_count(text: str) -> str:
    """Counts words, characters, and sentences in the given text.
    
    Args:
        text: The text to analyze
    """
    words = len(text.split())
    chars = len(text)
    sentences = text.count('.') + text.count('!') + text.count('?') or 1
    return f"Words: {words}, Characters: {chars}, Sentences: {sentences}"

# Test manually
print("calculate('245 * 38'):", calculate.invoke("245 * 38"))
print("get_word_count:", get_word_count.invoke("AI agents are the future."))
print()

# Inspect the tool schema — this is what the LLM sees
print("Tool name:", calculate.name)
print("Tool description:", calculate.description)

---
## 7. LangChain Agents — The brain picks the tools

Now we bring everything together. A LangChain agent:
1. Receives a question
2. Decides which tool to call (or answer directly)
3. Calls the tool and reads the result
4. Decides if it needs more tools, or gives the final answer

### 7.1 Creating the agent

In [ ]:
from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# The agent needs a special prompt with a placeholder for intermediate steps
agent_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI assistant. Use the available tools when needed. Be concise."),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

# Define tools
tools = [calculate, search_web, get_word_count]

# Create the agent (the brain)
agent = create_tool_calling_agent(llm, tools, agent_prompt)

# Create the executor (the runtime loop)
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,     # SEE EVERY STEP
    max_iterations=5  # safety limit
)

print(f"Agent ready with {len(tools)} tools: {[t.name for t in tools]}")

### 7.2 Testing the agent — watch which tool it picks

Same agent, different questions — different tools chosen automatically.

In [ ]:
# Query 1: Should use calculate
print("=" * 60)
print("Query 1: Math question")
print("=" * 60)
result = agent_executor.invoke({"input": "What is 1947 * 77 + 2025?"})
print(f"\nFinal Answer: {result['output']}")

In [ ]:
# Query 2: Should use search_web
print("=" * 60)
print("Query 2: Needs current information")
print("=" * 60)
result = agent_executor.invoke({"input": "What is the current population of Visakhapatnam?"})
print(f"\nFinal Answer: {result['output']}")

In [ ]:
# Query 3: Should use get_word_count
print("=" * 60)
print("Query 3: Text analysis")
print("=" * 60)
result = agent_executor.invoke({
    "input": "Count the words in: Agentic AI is transforming how we build software applications in 2026."
})
print(f"\nFinal Answer: {result['output']}")

In [ ]:
# Query 4: Should NOT use any tool — answer directly
print("=" * 60)
print("Query 4: General knowledge (no tool needed)")
print("=" * 60)
result = agent_executor.invoke({"input": "What is the capital of India?"})
print(f"\nFinal Answer: {result['output']}")

**Look at the verbose output above.** For each question, the agent:
1. Read the question
2. Looked at all 3 tool descriptions
3. **Decided** which tool to use (or no tool at all)
4. Called the tool and read the result
5. Generated the final answer

This is the same ReAct pattern from Smolagents, now in LangChain's framework.

---

## 8. Putting it all together — Agent + Structured Output

Let's combine agents with output parsers for a complete application:

In [ ]:
import json
from langchain_core.output_parsers import JsonOutputParser

# Step 1: Use the agent to research
research_result = agent_executor.invoke({
    "input": "Search for latest news about AI agents in India."
})
print("=== Agent Research ===")
print(research_result['output'][:300], "...")
print()

# Step 2: Use a chain to structure the output
structure_prompt = ChatPromptTemplate.from_messages([
    ("system", "Convert research findings to JSON. Respond ONLY with valid JSON, no markdown, no backticks."),
    ("human", "Structure these findings with keys: topic, summary, key_points (list of 3).\n\nFindings: {findings}")
])

structure_chain = structure_prompt | llm | JsonOutputParser()

structured = structure_chain.invoke({"findings": research_result['output']})
print("=== Structured Output ===")
print(json.dumps(structured, indent=2))

---
## 9. Your turn — Build a custom chain + agent

**Challenge (10 minutes):** Build one of these:

**Option A: Analyze and Improve chain**
- Step 1: Analyze a paragraph for readability (use get_word_count tool)
- Step 2: Ask the LLM to rewrite it at a simpler reading level

**Option B: Add a custom tool to the agent**
- Create a new `@tool` (BMI calculator, EMI calculator, temperature converter, etc.)
- Add it to the agent's tool list
- Test with 3 different queries

**Option C: Build a smarter router**
- Add a 4th category to the `RunnableBranch` (e.g., "sales" questions)
- Test with questions from all 4 categories

In [ ]:
# YOUR CODE HERE

# Option A skeleton:
# analyze_prompt = ChatPromptTemplate.from_messages([...])
# simplify_prompt = ChatPromptTemplate.from_messages([...])
# full_chain = analyze_chain | RunnableLambda(...) | simplify_chain

# Option B skeleton:
# @tool
# def my_tool(input_val: str) -> str:
#     """Description"""
#     ...
# tools = [calculate, search_web, get_word_count, my_tool]
# agent = create_tool_calling_agent(llm, tools, agent_prompt)
# executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

# Option C skeleton:
# sales_chain = sales_prompt | llm | StrOutputParser()
# router = RunnableBranch(
#     (billing_cond, billing_chain),
#     (tech_cond, technical_chain),
#     (sales_cond, sales_chain),
#     general_chain
# )

---
## 10. Quick Reference — Old vs New LangChain

Most online tutorials use deprecated patterns. Here's what to use in 2026:

| OLD (Deprecated in 0.3+) | NEW (Use This) |
|---|---|
| `LLMChain(llm=llm, prompt=prompt)` | `prompt \| llm \| parser` |
| `SequentialChain` | LCEL pipe chain |
| `RouterChain` | `RunnableBranch` |
| `chain.run("input")` | `chain.invoke({"key": "input"})` |
| `from langchain.chat_models import ...` | `from langchain_openai import ...` |
| `initialize_agent()` | `create_tool_calling_agent` + `AgentExecutor` |
| `ConversationChain` | `RunnableWithMessageHistory` |

**Rule of thumb:** If you see `LLMChain` in a tutorial, that tutorial is outdated.

---
## 11. Day 4 Wrap-up

### What you built today:
- Prompt Templates — reusable prompts with variables
- Output Parsers — StrOutputParser, JsonOutputParser, PydanticOutputParser
- LCEL pipe chains — `prompt | llm | parser`
- Sequential chains — multi-step pipelines with RunnableLambda glue
- RunnableBranch — routing different inputs to specialized chains
- LangChain agents — `create_tool_calling_agent` + `AgentExecutor`
- Combined agent + structured output in one application

### The mental model:
```
User Input
    |
PromptTemplate (format with variables)
    |
LLM (generate response)
    |
OutputParser (structure the output)
    |
[If agent] -> Tool Selection -> Tool Execution -> Back to LLM
    |
Final Output
```

### Key LangChain pattern to remember:
```python
chain = prompt | llm | parser          # basic chain
chain = step1 | glue | step2           # sequential
chain = RunnableBranch(conditions...)   # routing
chain = AgentExecutor(agent, tools)     # agent
```

### Tomorrow — Day 5: LangChain Agents & Tools (Deep Dive)
We go deeper into agent patterns: custom tool creation, AgentExecutor internals, error handling, verbose debugging, and building a multi-tool research assistant.

---

### Resources:
- [LangChain LCEL docs](https://python.langchain.com/docs/concepts/lcel/)
- [LangChain agents](https://python.langchain.com/docs/how_to/#agents)
- [LangChain output parsers](https://python.langchain.com/docs/concepts/output_parsers/)